In [1]:
from google.colab import files

uploaded = files.upload()

Saving time_series_60min_singleindex.csv to time_series_60min_singleindex.csv


In [17]:
import pandas as pd
import numpy as np

df = pd.read_csv('time_series_60min_singleindex.csv')

energy = pd.DataFrame()

energy['date'] = pd.to_datetime(df['utc_timestamp'])
energy['price'] = df['DE_LU_price_day_ahead']

energy = energy.dropna()
energy = energy.set_index('date')

print(energy.shape)
energy.head()

(17540, 1)


,price
date,
2018-09-30 23:00:00+00:00,56.10
2018-10-01 00:00:00+00:00,51.41
2018-10-01 01:00:00+00:00,47.38
2018-10-01 02:00:00+00:00,47.59
2018-10-01 03:00:00+00:00,51.61


In [18]:
#returns
energy['return'] = energy['price'].pct_change()

energy = energy.replace(
    [np.inf, -np.inf],
    np.nan
)

energy = energy.dropna()

print(energy.shape)

energy['return'].describe()

(17533, 2)


,return
count,17533.000000
mean,-0.076576
std,40.502677
min,-3744.000000
25%,-0.070117
50%,-0.010846
75%,0.061641
max,1303.000000


In [19]:
#stylized facts
from scipy.stats import skew, kurtosis

print(
    "Skew:",
    skew(energy['return'])
)

print(
    "Kurtosis:",
    kurtosis(energy['return'])
)

Skew: -55.72422324862008
Kurtosis: 5017.58000563617


In [20]:
#ARCH Test
from statsmodels.stats.diagnostic import het_arch

test = het_arch(
    energy['return']
)

print(test)

(np.float64(0.04480692473037784), np.float64(0.9999999999538376), 0.004477891184673421, 0.9999999999539291)


Βλέπουμε:

mean = -0.076
std = 40.5
min = -3744
max = 1303
skew = -55.7
kurtosis = 5017

Αυτά είναι εξωπραγματικά μεγάλα.

Ο λόγος είναι ότι υπάρχουν πολλές ώρες με τιμές πολύ κοντά στο μηδέν ή αρνητικές τιμές.

Παράδειγμα:

Price(t-1)=0.01
Price(t)=13

τότε:

return=(13-0.01)/0.01=1299

οπότε καταστρέφεται όλη η κατανομή.

Το σημαντικό εύρημα

Το ARCH test βγήκε:

p-value ≈ 1

που είναι επίσης ύποπτο και συμβαίνει ακριβώς επειδή τα returns έχουν γίνει ακραία.

Για το project θα χρησιμοποιήσουμε την κλασική προσέγγιση των electricity markets:
Price differences

In [21]:
energy = pd.DataFrame()

energy['date'] = pd.to_datetime(df['utc_timestamp'])
energy['price'] = df['DE_LU_price_day_ahead']

energy = energy.dropna()
energy = energy.set_index('date')

energy['return'] = energy['price'].diff()

energy = energy.dropna()

print(
    energy['return']
    .describe()
)

count    17539.000000
mean        -0.001237
std          5.551050
min       -101.910000
25%         -2.445000
50%         -0.320000
75%          2.095000
max        109.230000
Name: return, dtype: float64


In [22]:
from scipy.stats import skew, kurtosis

print(
    "Skew:",
    skew(energy['return'])
)

print(
    "Kurtosis:",
    kurtosis(energy['return'])
)

Skew: 0.48566319834168453
Kurtosis: 26.21171337626122


In [23]:
from statsmodels.stats.diagnostic import het_arch

print(
    het_arch(
        energy['return']
    )
)

(np.float64(3765.0995885891107), np.float64(0.0), 479.2029339170656, 0.0)


1. Skewness = 0.49
Υπάρχει μικρή θετική ασυμμετρία.

Αυτό σημαίνει ότι υπάρχουν περισσότεροι θετικοί ακραίοι κραδασμοί από ό,τι θα περίμενε μια κανονική κατανομή.

2. Kurtosis = 26.2

Αυτό είναι εξαιρετικά σημαντικό.

Η κανονική κατανομή έχει:

Kurtosis = 3

Εσύ έχεις:

Kurtosis = 26.2

που σημαίνει:

fat tails, δηλαδή ακραία γεγονότα εμφανίζονται πολύ συχνότερα από ό,τι προβλέπει η κανονική κατανομή.

Αυτό είναι χαρακτηριστικό των:

financial markets,
electricity markets,
commodity markets.
3. ARCH test

Έχουμε:

p-value = 0

δηλαδή απορρίπτουμε πλήρως τη μηδενική υπόθεση.

Με απλά λόγια:

η μεταβλητότητα εμφανίζεται σε clusters.

In [26]:
!pip install arch

#GARCH(1,1)
from arch import arch_model

garch = arch_model(
    energy['return'],
    mean='Constant',
    vol='GARCH',
    p=1,
    q=1,
    dist='t'
)

garch_fit = garch.fit()

print(garch_fit.summary())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 14.9 MB/s eta 0:00:00
Iteration:      1,   Func. Count:      7,   Neg. LLF: 560199.4189588731
Iteration:      2,   Func. Count:     16,   Neg. LLF: 279491.29844510445
Iteration:      3,   Func. Count:     23,   Neg. LLF: 70357.54726588774
Iteration:      4,   Func. Count:     30,   Neg. LLF: 67965.81476287656
Iteration:      5,   Func. Count:     40,   Neg. LLF: 66125.15954491968
Iteration:      6,   Func. Count:     47,   Neg. LLF: 50560.33669464038
Iteration:      7,   Func. Count:     53,   Neg. LLF: 64228.15637981024
Iteration:      8,   Func. Count:     60,   Neg. LLF: 65806.19242493853
Iteration:      9,   Func. Count:     67,   Neg. LLF: 50635.44189670287
Iteration:     10,   Func. Count:     74,   Neg. LLF: 50273.356743132405
Iteration:     11,   Func. Count:     80,   Neg. LLF: 50232.88825909324
Iteration:     12,   Func. Count:     86,   Neg. LLF: 50205.320732868735
Iteration:     13,   Func. Count:     92,   Neg. LLF

Γιατί χρησιμοποιούμε dist='t';

Επειδή μόλις αποδείξαμε ότι έχουμε:

Kurtosis = 26

δηλαδή fat tails.

Η Student-t κατανομή είναι σχεδόν πάντα καλύτερη από την Gaussian στις αγορές ενέργειας.

Τι μας λέει το μοντέλο;
1. Volatility persistence

Το πρώτο πράγμα που κοιτάμε πάντα είναι:

α + β

Εδώ έχουμε:

0.891 + 0.109 = 1.000

Αυτό είναι εξαιρετικά ενδιαφέρον.

Σημαίνει:

η μεταβλητότητα στην αγορά ηλεκτρικής ενέργειας είναι σχεδόν πλήρως persistent.

Με άλλα λόγια:

ένα μεγάλο shock σήμερα
↓
επηρεάζει τη μεταβλητότητα για πολλές ώρες ή ημέρες
2. Shock reaction

Το:

alpha = 0.891

είναι πολύ υψηλό.

Αυτό σημαίνει:

η αγορά αντιδρά εξαιρετικά έντονα σε νέα πληροφορία.

Αυτό είναι χαρακτηριστικό των electricity markets λόγω:

renewable intermittency,
demand spikes,
transmission constraints.
3. Heavy tails

Το:

nu = 3.66

είναι ίσως το πιο ενδιαφέρον αποτέλεσμα.

Για Student-t:

ν → ∞

πλησιάζει την Gaussian.

Εσύ έχεις:

ν = 3.66

δηλαδή:

εξαιρετικά βαριές ουρές.

Αυτό επιβεβαιώνει πλήρως το αποτέλεσμα της kurtosis=26.

In [27]:
#Model 2- EGARCH(1,1)
egarch = arch_model(
    energy['return'],
    mean='Constant',
    vol='EGARCH',
    p=1,
    q=1,
    dist='t'
)

egarch_fit = egarch.fit()

print(egarch_fit.summary())

Iteration:      1,   Func. Count:      7,   Neg. LLF: 346717.00895076914
Iteration:      2,   Func. Count:     20,   Neg. LLF: 327194.50550229504
Iteration:      3,   Func. Count:     30,   Neg. LLF: 366481.96833794034
Iteration:      4,   Func. Count:     39,   Neg. LLF: 73379.21213584727
Iteration:      5,   Func. Count:     46,   Neg. LLF: 65767.39914359624
Iteration:      6,   Func. Count:     53,   Neg. LLF: 63851.08635958007
Iteration:      7,   Func. Count:     60,   Neg. LLF: 58752.02896824934
Iteration:      8,   Func. Count:     67,   Neg. LLF: 63416.54254112425
Iteration:      9,   Func. Count:     74,   Neg. LLF: 63349.77277821703
Iteration:     10,   Func. Count:     81,   Neg. LLF: 63262.73166917103
Iteration:     11,   Func. Count:     88,   Neg. LLF: 63150.739643912944
Iteration:     12,   Func. Count:     95,   Neg. LLF: 54344.01315752279
Iteration:     13,   Func. Count:    102,   Neg. LLF: 50170.96484567476
Iteration:     14,   Func. Count:    108,   Neg. LLF: 50206.

Ερμηνεία EGARCH
Mean
μ = -0.620

Παρόμοιο με το GARCH.

Volatility persistence

Έχουμε:

beta = 0.498

που δείχνει μέτρια persistence.

Shock response
alpha = 1.220

Πολύ μεγάλο.

Αυτό σημαίνει ότι η μεταβλητότητα επηρεάζεται έντονα από νέα shocks.

Heavy tails
nu = 3.17

Ακόμη μικρότερο από το GARCH:

GARCH : 3.66
EGARCH: 3.17

Άρα το μοντέλο συνεχίζει να βλέπει πολύ βαριές ουρές.

Το σημαντικό εύρημα

Παρατηρώ ότι δεν εμφανίζεται:

gamma[1]

Η παράμετρος αυτή είναι η ασυμμετρία (leverage effect).

Αυτό σημαίνει ότι το συγκεκριμένο EGARCH specification δεν εκτίμησε ασύμμετρο όρο ή ότι δεν τον ζητήσαμε.

In [28]:
#GJR-GARCH
gjr = arch_model(
    energy['return'],
    mean='Constant',
    vol='GARCH',
    p=1,
    o=1,      # asymmetry term
    q=1,
    dist='t'
)

gjr_fit = gjr.fit()

print(gjr_fit.summary())

Iteration:      1,   Func. Count:      8,   Neg. LLF: 166201.63336964458
Iteration:      2,   Func. Count:     19,   Neg. LLF: 176319.67293440012
Iteration:      3,   Func. Count:     27,   Neg. LLF: 66130.8080365581
Iteration:      4,   Func. Count:     35,   Neg. LLF: 65859.3256405964
Iteration:      5,   Func. Count:     43,   Neg. LLF: 51059.299689544256
Iteration:      6,   Func. Count:     51,   Neg. LLF: 65712.51470061812
Iteration:      7,   Func. Count:     59,   Neg. LLF: 50865.61311805324
Iteration:      8,   Func. Count:     67,   Neg. LLF: 65473.60079891846
Iteration:      9,   Func. Count:     75,   Neg. LLF: 50090.964180338924
Iteration:     10,   Func. Count:     82,   Neg. LLF: 50034.48055899313
Iteration:     11,   Func. Count:     89,   Neg. LLF: 50789.36372507749
Iteration:     12,   Func. Count:     97,   Neg. LLF: 49985.3301365055
Iteration:     13,   Func. Count:    104,   Neg. LLF: 49980.405679093084
Iteration:     14,   Func. Count:    111,   Neg. LLF: 49971.38

Πρώτο συμπέρασμα

Το GJR-GARCH είναι ξεκάθαρα το καλύτερο μοντέλο:

AIC improvement ≈ 361 μονάδες

Αυτό είναι τεράστια βελτίωση.

Άρα:

Η μεταβλητότητα της γερμανικής αγοράς ηλεκτρισμού είναι ασύμμετρη.

Ανάλυση των παραμέτρων
ω = 6.11

Είναι το long-run variance component.

α = 1.00
alpha = 1

Αυτό είναι εξαιρετικά υψηλό.

Σημαίνει:

η αγορά αντιδρά σχεδόν άμεσα και πολύ έντονα στα νέα shocks.

γ = -0.63

Αυτό είναι το σημαντικότερο αποτέλεσμα όλου του project.

Έχουμε:

gamma = -0.63
p-value << 0.001

δηλαδή ισχυρή ασυμμετρία.

Πώς ερμηνεύεται το αρνητικό gamma;

Στις μετοχές συνήθως έχουμε:

gamma > 0

(οι αρνητικές αποδόσεις αυξάνουν τη μεταβλητότητα).

Στην ηλεκτρική ενέργεια όμως συχνά συμβαίνει το αντίθετο:

positive price shocks
       >
negative price shocks

Δηλαδή:

shortages,
demand spikes,
renewable production failures,

προκαλούν μεγαλύτερη μεταβλητότητα από ό,τι οι αρνητικές τιμές.

Αυτό είναι απολύτως συμβατό με τη βιβλιογραφία των power markets.

β = 0.20

Η persistence είναι αρκετά μικρότερη από το απλό GARCH.

Αυτό σημαίνει ότι μεγάλο μέρος της persistence που βλέπαμε πριν εξηγείται πλέον από την ασυμμετρία.

ν = 3.93

Παραμένουν εξαιρετικά βαριές ουρές:

Student-t ν ≈ 4

οπότε έχουμε ισχυρή ένδειξη για:

price spikes,
extreme events,
market stress periods.

The German electricity market exhibits significant volatility clustering, heavy tails and asymmetric volatility dynamics. Among the estimated models, the GJR-GARCH specification provided the best fit according to the AIC criterion, indicating that positive price shocks generate stronger volatility responses than negative shocks.